<a href="https://colab.research.google.com/github/ekc2024/ScamGuard-MY/blob/main/Experian_PDPA_Credit_Analyzer_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experian Credit Report Analyzer (PDPA Compliant)
This notebook extracts unstructured text from a Malaysian credit report or spreadsheet, redacts Personally Identifiable Information (PII) like NRICs, emails, and phone numbers for **PDPA compliance**, flags potential commercial risks, and presents an AI-powered risk dashboard.

**Supports both PDF and spreadsheet (.xlsx / .csv) input.** You can generate a mock sample to test with, or upload your own file.

### 1. Install Dependencies
- `pdfplumber` — text extraction from PDFs
- `openpyxl` — required for reading `.xlsx` files with pandas
- `reportlab` — only needed if you want to generate the mock sample PDF in Step 3

In [1]:
!pip install pdfplumber openpyxl reportlab -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 71.6 MB/s eta 0:00:00


### 2. Define the PII Masking, Extraction & Risk Detection Engine
This single cell defines everything: PII masking, PDF extraction, spreadsheet extraction, the format router, and the risk-detection pipeline. Run this cell once, then don't touch it again unless you're changing the underlying logic.

In [2]:
import re
import pdfplumber
import pandas as pd


def mask_experian_pii(text: str) -> str:
    """Masks Malaysian NRICs, Emails, and Phone Numbers for PDPA Compliance."""
    nric_pattern = r"\b\d{6}-\d{2}-\d{4}\b"
    email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"
    phone_pattern = r"\b01[0-9][-\s]?\d{3,4}[-\s]?\d{4}\b"

    masked = re.sub(nric_pattern, "[REDACTED_NRIC]", text)
    masked = re.sub(email_pattern, "[REDACTED_EMAIL]", masked)
    masked = re.sub(phone_pattern, "[REDACTED_PHONE]", masked)

    return masked


def extract_from_spreadsheet(file_path: str) -> str:
    """Extracts and flattens spreadsheet data (xlsx or csv) into text for the same pipeline."""
    if file_path.lower().endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)

    lines = []
    for _, row in df.iterrows():
        line = " | ".join(f"{col}: {val}" for col, val in row.items())
        lines.append(line)
    return "\n".join(lines)


def extract_any(file_path: str) -> str:
    """Router: sends the file to the right extractor based on its extension, then masks PII."""
    if file_path.lower().endswith((".xlsx", ".xls", ".csv")):
        raw_text = extract_from_spreadsheet(file_path)
    else:
        raw_text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    raw_text += text + "\n"
    return mask_experian_pii(raw_text)


def detect_risks(sanitized_text: str, risk_keywords=None) -> list:
    """Flags lines containing risk-related keywords."""
    if risk_keywords is None:
        risk_keywords = ["lawsuit", "liabilities", "deteriorating", "unpaid", "debt", "risk"]

    detected_risks = []
    for line in sanitized_text.split('\n'):
        if any(keyword in line.lower() for keyword in risk_keywords):
            if line.strip() and line.strip() not in detected_risks:
                detected_risks.append(line.strip())
    return detected_risks


def analyze_credit_report(file_path: str):
    """Runs the full pipeline: extract (PDF or spreadsheet) -> mask PII -> detect risks -> print results."""
    try:
        sanitized_text = extract_any(file_path)
    except FileNotFoundError:
        print(f"Error: Could not find {file_path}")
        print("Please make sure you have generated or uploaded the file first.")
        return

    detected_risks = detect_risks(sanitized_text)

    print("--- 1. PDPA SANITIZED OUTPUT ---")
    print(sanitized_text)

    print("\n--- 2. DETECTED CREDIT RISKS ---")
    for idx, risk in enumerate(detected_risks, 1):
        print(f"[{idx}] {risk}")

### 3. (Option A) Generate a Mock Experian Credit Report
Since real credit reports are strictly confidential, run this cell to create a simulated **PDF** file containing artificial Malaysian PII and risk indicators to test the engine.

**Skip this if you'd rather upload your own PDF or spreadsheet in Step 4.**

In [3]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def create_mock_pdf(filename="Credit_Assessment_Pinnacle.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "EXPERIAN COMMERCIAL CREDIT ASSESSMENT")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 700, "Company Name: Pinnacle Tech Solutions")
    c.drawString(100, 680, "Director Name: Ahmad Razak")
    c.drawString(100, 660, "Director NRIC: 880412-14-5531")
    c.drawString(100, 640, "Contact Email: ahmad.razak@pinnacle.com.my")
    c.drawString(100, 620, "Mobile Phone: 012-3456789")

    c.drawString(100, 580, "FINANCIAL SUMMARY & RISK METRICS:")
    c.drawString(100, 560, "- The entity shows flags of deteriorating capital reserves.")
    c.drawString(100, 540, "- High risk exposure detected due to severe unpaid supplier invoices.")
    c.drawString(100, 520, "- Active lawsuit filed by major vendor on 15/04/2026.")
    c.drawString(100, 500, "- Total outstanding liabilities exceed RM 450,000.")

    c.save()
    print(f"\u2705 Mock PDF '{filename}' generated successfully.")

create_mock_pdf()
sample_file_path = "Credit_Assessment_Pinnacle.pdf"

✅ Mock PDF 'Credit_Assessment_Pinnacle.pdf' generated successfully.


### 4. (Option B) Upload Your Own PDF or Spreadsheet
Run this cell instead of Step 3 if you want to test with a real file — a PDF credit report, or an `.xlsx`/`.csv` spreadsheet of financial data. This opens a native Colab file picker.

**If you already ran Step 3, running this too will simply overwrite `sample_file_path` with your uploaded file.**

In [ ]:
from google.colab import files

print("\U0001F4E4 Please select a PDF or spreadsheet (.xlsx / .csv) file to upload...")
uploaded = files.upload()

if uploaded:
    sample_file_path = list(uploaded.keys())[0]
    print(f"\u2705 '{sample_file_path}' uploaded successfully.")
else:
    print("\u26A0\uFE0F No file uploaded \u2014 keeping the existing sample_file_path (if set).")

📤 Please select a PDF or spreadsheet (.xlsx / .csv) file to upload...


### 5. AI Analysis Simulation + Dashboard Renderer

> ⚠️ **Note:** `simulate_llm_analysis()` below currently returns a **hardcoded mock response** — it does not actually read the sanitized text. This is fine for testing the pipeline/UI, but for a real submission you should replace it with a genuine LLM API call (see Step 7) so the risk score and summary actually reflect the uploaded document.

In [ ]:
from IPython.display import display, HTML

def simulate_llm_analysis(sanitized_text: str) -> dict:
    """
    Simulates sending the PDPA-compliant text to an LLM (e.g. Claude or Gemini)
    using a structured prompt to extract Experian-style risk metrics.

    NOTE: This is a MOCK response for demo purposes — it does not vary based on
    the actual sanitized_text passed in. Replace with a real API call (see Step 7)
    for production use.
    """
    prompt = f"""
    Context: You are a corporate financial risk analyst.
    Task: Review the following sanitized credit report and extract key insights.
    Format your response as a JSON object containing a Risk Score (1-100), a Summary,
    and a list of Actionable Recommendations.
    Data: {sanitized_text}
    """

    mock_llm_response = {
        "risk_score": 35,
        "risk_category": "Medium to High Risk",
        "executive_summary": "The entity exhibits significant financial distress indicators. Deteriorating capital reserves and severe unpaid supplier invoices suggest severe cash flow bottlenecks.",
        "actionable_recommendations": [
            "Request immediate clarification on the active vendor lawsuit filed in April 2026.",
            "Implement stricter net-15 payment terms for any future engagements.",
            "Require a guarantor or upfront deposit before extending further credit."
        ],
        "explainability_flag": "Risk score heavily weighted by the presence of active litigation and liabilities exceeding RM 450,000."
    }

    return mock_llm_response


def render_financial_dashboard(ai_insights: dict):
    """Renders the AI insights into a clean, intuitive HTML dashboard directly in Colab."""
    score_color = "red" if ai_insights["risk_score"] < 50 else "green"

    dashboard_html = f"""
    <div style="font-family: sans-serif; border: 2px solid #e5e7eb; border-radius: 10px; padding: 20px; max-width: 800px;">
        <h2 style="color: #1f2937; border-bottom: 2px solid #e5e7eb; padding-bottom: 10px;">\U0001F4CA AI Credit Risk Dashboard</h2>

        <div style="display: flex; justify-content: space-between; margin-top: 20px;">
            <div style="background-color: #f9fafb; padding: 15px; border-radius: 8px; width: 45%;">
                <h3 style="margin: 0; color: #4b5563;">Experian Risk Score</h3>
                <h1 style="margin: 10px 0; font-size: 48px; color: {score_color};">{ai_insights['risk_score']}/100</h1>
                <p style="margin: 0; font-weight: bold; color: {score_color};">{ai_insights['risk_category']}</p>
            </div>

            <div style="width: 50%;">
                <h3 style="margin-top: 0; color: #4b5563;">Executive Summary</h3>
                <p style="color: #374151; line-height: 1.5;">{ai_insights['executive_summary']}</p>
                <p style="font-size: 12px; color: #6b7280;"><i>Model Rationale: {ai_insights['explainability_flag']}</i></p>
            </div>
        </div>

        <h3 style="color: #4b5563; margin-top: 25px;">\u26A1 Actionable Recommendations</h3>
        <ul style="color: #374151; line-height: 1.6;">
            {"".join(f"<li>{item}</li>" for item in ai_insights['actionable_recommendations'])}
        </ul>

        <p style="font-size: 11px; color: #9ca3af; margin-top: 20px; border-top: 1px solid #f3f4f6; padding-top: 10px;">
            \U0001F512 PDPA Notice: This document was processed in-memory only. Detected NRICs, emails, and phone numbers were redacted before analysis and are not stored or logged anywhere.
        </p>
    </div>
    """
    display(HTML(dashboard_html))

### 6. Run the Full Pipeline
This extracts text from `sample_file_path` (PDF or spreadsheet), redacts PII, flags risk lines, prints the sanitized output, then sends it through the (currently mock) AI analysis step and renders the dashboard.

Set `sample_file_path` above (Step 3 or Step 4) before running this.

In [ ]:
sanitized_text = extract_any(sample_file_path)
risks = detect_risks(sanitized_text)

print("--- 1. PDPA SANITIZED OUTPUT ---")
print(sanitized_text)

print("\n--- 2. DETECTED CREDIT RISKS ---")
for idx, risk in enumerate(risks, 1):
    print(f"[{idx}] {risk}")

print("\nSending sanitized data to AI Engine...\n")
ai_results = simulate_llm_analysis(sanitized_text)
render_financial_dashboard(ai_results)

### 7. (For Your Final Submission) Replace the Mock LLM Call

Right now `simulate_llm_analysis()` returns a **fixed dictionary** no matter what's in `sanitized_text`. If judges upload a different file and see the same score and summary every time, that will hurt you on the "accuracy and quality of insights" success criterion.

To fix this, replace the body of `simulate_llm_analysis()` with a real API call, e.g.:

```python
import anthropic
import json

client = anthropic.Anthropic(api_key="YOUR_API_KEY")

def simulate_llm_analysis(sanitized_text: str) -> dict:
    prompt = f'''
    Context: You are a corporate financial risk analyst.
    Task: Review the following sanitized credit report and extract key insights.
    Respond ONLY with a JSON object (no markdown, no preamble) with keys:
    risk_score (1-100 integer), risk_category (string), executive_summary (string),
    actionable_recommendations (list of strings), explainability_flag (string).
    Data: {sanitized_text}
    '''

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )

    raw_text = response.content[0].text.strip()
    raw_text = raw_text.replace("```json", "").replace("```", "").strip()
    return json.loads(raw_text)
```

You'd need to `!pip install anthropic` and store your API key securely (e.g. via Colab's Secrets manager, `google.colab.userdata`) rather than hardcoding it. Swap in Gemini's SDK instead if that's the model your team is using — the surrounding pipeline (extraction, masking, dashboard rendering) stays exactly the same either way.